<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/3_dise%C3%B1o_entrenamiento_evaluacion/3_2_modelo_xx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.2. Primer modelo

## 0. Clonado de repositorio, importación de librerías y carga del dataset

### Clonado de repositorio e importación de librerías

In [ ]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit

!git clone https://github.com/GUNAPILLCO/neural_profit.git

Cloning into 'neural_profit'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 276 (delta 8), reused 10 (delta 4), pack-reused 257 (from 1)
Receiving objects: 100% (276/276), 176.68 MiB | 34.00 MiB/s, done.
Resolving deltas: 100% (149/149), done.
Updating files: 100% (49/49), done.


In [ ]:
import sys

#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")

!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
import seaborn as sns

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


### Carga de datasets train, valid y test

In [ ]:
def load_df():
    """
    Función para cargar un archivo Parquet desde el repositorio clonado
    """
    # Definir la URL del archivo Parquet en GitHub
    df_path_mnq = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_model.parquet'
    df_path_factores = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/df_factores.parquet'
    df_path_train = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_train.parquet'
    df_path_valid = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_valid.parquet'
    df_path_test = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_test.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)
    df_factores = pd.read_parquet (df_path_factores)
    df_train = pd.read_parquet(df_path_train)
    df_valid = pd.read_parquet(df_path_valid)
    df_test = pd.read_parquet(df_path_test)

    return df_model, df_factores, df_train, df_valid, df_test

In [ ]:
mnq_model, indicadores_tecnicos, mnq_train, mnq_valid, mnq_test = load_df()

### Información del dataset

In [ ]:
def info_dataset (df): # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"Cantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['target_return_30']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"Valores por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo
  print(f"Hora diaria de inicio {primer_hora}")
  print(f"Hora diaria de final {ultima_hora}")
  print(f"Zona horaria: {zona_horaria}")

In [ ]:
info_dataset(mnq_train)

Cantidad de días: 917
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [ ]:
info_dataset(mnq_valid)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [ ]:
info_dataset(mnq_test)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


# 1. Modelos

**🔹Modelos tabulares (altamente recomendados)**

- ✅Random Forest / XGBoost / LightGBM
  - Aprovechan los alpha factors.
  - Robustez frente a outliers.
  - Fácil interpretación de importancia de variables.

- ✅MLP (Perceptrón multicapa)
  - Rápido de entrenar.
  - Bueno si normalizás correctamente los factores.

- ⚠️ Regresiones lineales (ridge/lasso)
  - Útiles como baseline o si buscás interpretabilidad.
  - Evitar si hay mucha multicolinealidad.

  **✔️ Tu dataset mnq_model es ideal para todos los anteriores.**


**🔹Modelos secuenciales / deep learning**

- ✅ LSTM / GRU: Recomendados si querés explotar la secuencia temporal minuto a minuto por jornada.
  - Tu dataset tiene 451 registros por día, perfecto para construir ventanas tipo (60, n_features) → target_return_30.

- ✅ CNN-1D / TCN / Transformer:
  - Requieren más entrenamiento, pero pueden capturar patrones complejos.
  - Buen rendimiento si tenés suficientes datos (¡lo tenés!).

  **✔️ Estos modelos requieren que estructures el dataset en formato secuencial. No es inmediato, pero tu estructura diaria fija lo facilita.**

**🔹 Modelos híbridos / avanzados**

- ✅ Autoencoder + MLP/LSTM: útil si querés reducir dimensionalidad y eliminar redundancias entre factores correlacionados.
- ✅ Stacking: combinar Random Forest + XGBoost + MLP puede dar muy buenos resultados.
- ✅ SHAP + XGBoost: si querés explicar decisiones del modelo.

<br><br>

**⭐ Recomendación final (ajustada a tus datos)**

| Modelo               | Justificación                                                                 |
|----------------------|------------------------------------------------------------------------------|
| XGBoost / LightGBM   | Tabular, aprovechan todos tus factores alpha, rápidos y robustos.           |
| Random Forest        | Ideal para análisis de importancia de factores y como baseline.             |
| MLP                  | Simple, pero potente con buena normalización.                               |
| LSTM / GRU           | Perfecto si estructurás las secuencias por jornada de 60 minutos.           |
| Stacking (Ensemble)  | Alta precisión si combinás modelos tabulares y profundos.                   |

- Random Forest
- XGBoost
- LightGBM

# 2. Modelos XGBoost / LightGBM

Cantidad de fechas únicas: 1311
Promedio de valores válidos de 'target_return_30' por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York
